<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Logistic Regression Practice

---

# Titanic
On April 15, 1912, the RMS Titanic sank after colliding with an iceberg, killing over 1,500 of its 2,224 passengers and crew. Survival wasn't random — factors like class, sex, and age all played a role in who made it into a lifeboat.

You will be predicting `Survived`: whether a passenger survived (1) or did not (0), using their ticket and demographic information.

# Imports

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score, recall_score, precision_score, f1_score

### 1. Load the data

Read in the `titanic.csv` data.

In [2]:
# TODO:
titanic = pd.read_csv('./data/titanic.csv')
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### 2. Drop columns that won’t help the model

> `PassengerId` is just a row ID, `Name` and `Ticket` are high-cardinality free text, and `Cabin` is missing for the vast majority of passengers.

In [3]:
# TODO:
titanic = titanic.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

### 3. Handle missing values

> Check for missing values. `Age` and `Embarked` both have null values, fill them in (median for `Age` and the most common value for `Embarked`).

In [4]:
titanic['Age'].isnull().sum()

np.int64(177)

In [5]:
# TODO:
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
titanic['Age'].isnull().sum()

np.int64(0)

In [6]:
titanic['Embarked'].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [7]:
titanic['Embarked'].isnull().sum()

np.int64(2)

In [8]:
titanic['Embarked'] = titanic['Embarked'].fillna('S')
titanic['Embarked'].isnull().sum()

np.int64(0)

# Modeling

### 4. Set up your target and features, and split into training and testing sets

Set `Survived` as your target, `y`. Everything else remaining is your feature matrix, `X`.

In [9]:
# TODO:
X = titanic.drop('Survived', axis = 1)
y = titanic['Survived']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 5. Establish a baseline

What accuracy would you get by always predicting the majority class? This is the number your model needs to beat.

In [10]:
# TODO:
print(titanic['Survived'].value_counts(normalize=True))

baseline_accuracy = titanic['Survived'].value_counts(normalize=True).max()
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64
Baseline Accuracy: 0.6162


### 6. Build a `Pipeline` with a `ColumnTransformer`

> Use a `ColumnTransformer` to scale your numeric features and encode your categorical features. Then chain it into a `Pipeline`.

In [11]:
# TODO:
numeric_features = ['Age', 'Fare']
categorical_features = ['Sex', 'Embarked']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ]
)

model_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression())
    ]
)

### 7. Fit your pipeline and evaluate train vs. test accuracy

Is there a meaningful gap between train and test performance? Is this model better than the null model?

In [13]:
# TODO:
model_pipeline.fit(X_train, y_train)

train_acc = model_pipeline.score(X_train, y_train)
test_acc = model_pipeline.score(X_test, y_test)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

Train Accuracy: 0.7823
Test Accuracy:  0.7765


### 8. Confusion matrix

> Generate a confusion matrix for your model's predictions on the test set. Which type of mistake is your model making more (predicting survival for someone who didn't, or the reverse?)

In [14]:
# TODO:
y_pred = model_pipeline.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix Layout:")
print(cm)

Confusion Matrix Layout:
[[87 18]
 [22 52]]


### 9. Classification report

> Print a classification report (precision, recall, f1-score) for the test set.

In [15]:
# TODO:
print(classification_report(y_test, y_pred, target_names=['Perished (0)', 'Survived (1)']))

              precision    recall  f1-score   support

Perished (0)       0.80      0.83      0.81       105
Survived (1)       0.74      0.70      0.72        74

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179



# Interpretation & Predictions

### 10. Pull out your feature names and coefficients

> Your `ColumnTransformer` renames and reorders your columns when it transforms `X`, the numeric columns keep their original names, but the one-hot encoded categorical columns get new names like `Sex_male`. Before you can interpret any coefficient, you need to know which coefficient belongs to which feature.
>
> 1. Get the one-hot encoded feature names out of your fitted `ColumnTransformer` using:
>
>    `.named_steps['preprocessing'].named_transformers_['onehotencoder'].get_feature_names_out(cat_cols)`.
> 
> 2. Combine these with your numeric column names, **in the same order your `ColumnTransformer` outputs them** (e.g. numeric columns first, then categorical).
> 
> 3. Pair these feature names with your fitted model's `.coef_[0]` values in a `pandas.Series` (feature names as the index), so you can look up any coefficient by name.

In [16]:
# TODO:
cat_features_out = model_pipeline.named_steps['preprocessor'] \
                                 .named_transformers_['cat'] \
                                 .get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(cat_features_out)

coefficients = pd.Series(model_pipeline.named_steps['classifier'].coef_[0], index=all_feature_names)
print(coefficients)

Age          -0.067793
Fare          0.460160
Sex_male     -2.347460
Embarked_Q   -0.548506
Embarked_S   -0.596336
dtype: float64


### 11. Interpret one coefficient

> Pick one feature and interpret its coefficient. Convert it from log-odds into an odds ratio (`np.exp(coef)`), and explain in a sentence what it does to the odds of survival.

In [17]:
# TODO:
odds_ratios = np.exp(coefficients)
print("Odds Ratios:")
print(odds_ratios)

# Example Interpretation template for a feature like 'Sex_male' (assuming it is < 1):
# "Holding all other features constant, being male multiplies the odds of survival by [Odds Ratio Value], making them less likely to survive."

Odds Ratios:
Age           0.934454
Fare          1.584328
Sex_male      0.095612
Embarked_Q    0.577812
Embarked_S    0.550826
dtype: float64


### 12. Generate predictions and predicted probabilities

> Use your model to generate `.predict()` and `.predict_proba()` on the test set. Pick one test observation and explain, in a sentence, what its predicted probability means.

In [18]:
# TODO:
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)

first_passenger_proba = y_proba[0][1] 
first_passenger_pred = y_pred[0]

print(f"First passenger predicted class: {first_passenger_pred}")
print(f"First passenger probability of survival: {first_passenger_proba:.4f}")

# Example Interpretation template:
# "The model estimates that this passenger has a [Probability * 100]% chance of surviving the Titanic disaster."

First passenger predicted class: 0
First passenger probability of survival: 0.2550


### 12. Try a different threshold

> `.predict()` uses a default threshold of 0.5 on `.predict_proba()` to decide the class. Try lowering it and recompute the confusion matrix and classification report at this new threshold. What trade-off do you see between precision and recall?

In [19]:
# TODO:
new_threshold = 0.3

y_pred_new = (y_proba[:, 1] >= new_threshold).astype(int)

print(f"--- Metrics at Threshold {new_threshold} ---")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_new))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_new, target_names=['Perished (0)', 'Survived (1)']))

--- Metrics at Threshold 0.3 ---

Confusion Matrix:
[[85 20]
 [19 55]]

Classification Report:
              precision    recall  f1-score   support

Perished (0)       0.82      0.81      0.81       105
Survived (1)       0.73      0.74      0.74        74

    accuracy                           0.78       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.78      0.78      0.78       179



### 13. Write a function to print classification metrics with explanations

> Write a function `print_classification_metrics(y_true, y_pred)` that calculates **accuracy**, **recall/sensitivity**, **precision**, **specificity**, and **F1 score**, and prints each metric alongside a clear explanation. Then call your function on your model's test set predictions.
>
> Use the confusion matrix layout below as your reference for which cells (`TP`, `TN`, `FP`, `FN`) each metric is built from.

**Accuracy** — Overall, how often is the classifier correct?

<table style="border: none">
<tr style="border: none">
    <td style="border: none; vertical-align: bottom; color: blue">n</td>
    <td><b>Predicted: Did Not Survive (0)</b></td>
    <td><b>Predicted: Survived (1)</b></td>
</tr>
<tr>
    <td><b>Actual: Did Not Survive (0)</b></td>
    <td style="text-align: center; background-color: red">TN</td>
    <td style="text-align: center">FP</td>
</tr>
<tr>
    <td><b>Actual: Survived (1)</b></td>
    <td style="text-align: center">FN</td>
    <td style="text-align: center; background-color: green">TP</td>
</tr>
</table>

$$\textbf{Accuracy} = \frac{TP + TN}{Total}$$

**Recall / Sensitivity / True Positive Rate (TPR)** — Among all passengers who actually survived, what proportion did we correctly detect?

<table style="border: none">
<tr style="border: none">
    <td style="border: none; vertical-align: bottom; color: blue">n</td>
    <td><b>Predicted: Did Not Survive (0)</b></td>
    <td><b>Predicted: Survived (1)</b></td>
</tr>
<tr>
    <td><b>Actual: Did Not Survive (0)</b></td>
    <td style="text-align: center">TN</td>
    <td style="text-align: center">FP</td>
</tr>
<tr>
    <td><b>Actual: Survived (1)</b></td>
    <td style="text-align: center; background-color: orchid">FN</td>
    <td style="text-align: center; background-color: green">TP</td>
</tr>
</table>

$$\textbf{Recall} = \frac{TP}{TP + FN}$$

**Precision** — Among all passengers predicted to survive, what proportion actually did?

<table style="border: none">
<tr style="border: none">
    <td style="border: none; vertical-align: bottom; color: blue">n</td>
    <td><b>Predicted: Did Not Survive (0)</b></td>
    <td><b>Predicted: Survived (1)</b></td>
</tr>
<tr>
    <td><b>Actual: Did Not Survive (0)</b></td>
    <td style="text-align: center">TN</td>
    <td style="text-align: center; background-color: orange">FP</td>
</tr>
<tr>
    <td><b>Actual: Survived (1)</b></td>
    <td style="text-align: center">FN</td>
    <td style="text-align: center; background-color: green">TP</td>
</tr>
</table>

$$\textbf{Precision} = \frac{TP}{TP + FP}$$

**Specificity / True Negative Rate (TNR)** — Among all passengers who did not actually survive, what proportion did we correctly detect?

<table style="border: none">
<tr style="border: none">
    <td style="border: none; vertical-align: bottom; color: blue">n</td>
    <td><b>Predicted: Did Not Survive (0)</b></td>
    <td><b>Predicted: Survived (1)</b></td>
</tr>
<tr>
    <td><b>Actual: Did Not Survive (0)</b></td>
    <td style="text-align: center; background-color: red">TN</td>
    <td style="text-align: center; background-color: orange">FP</td>
</tr>
<tr>
    <td><b>Actual: Survived (1)</b></td>
    <td style="text-align: center">FN</td>
    <td style="text-align: center">TP</td>
</tr>
</table>

$$\textbf{Specificity} = \frac{TN}{TN + FP}$$

**$F_1$ score** — The balance between precision and recall.

<table style="border: none">
<tr style="border: none">
    <td style="border: none; vertical-align: bottom; color: blue">n</td>
    <td><b>Predicted: Did Not Survive (0)</b></td>
    <td><b>Predicted: Survived (1)</b></td>
</tr>
<tr>
    <td><b>Actual: Did Not Survive (0)</b></td>
    <td style="text-align: center">TN</td>
    <td style="text-align: center; background-color: orange">FP</td>
</tr>
<tr>
    <td><b>Actual: Survived (1)</b></td>
    <td style="text-align: center; background-color: orchid">FN</td>
    <td style="text-align: center; background-color: green">TP</td>
</tr>
</table>

$$F_1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$


In [20]:
# TODO:
def print_classification_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"-> Explanation: Overall, how often is the classifier correct? (TP+TN)/Total\n")
    
    print(f"Precision: {precision:.4f}")
    print(f"-> Explanation: Out of all positive predictions, how many were actually positive? TP/(TP+FP)\n")
    
    print(f"Recall / Sensitivity: {recall:.4f}")
    print(f"-> Explanation: Out of all actual positive cases, how many did we catch? TP/(TP+FN)\n")
    
    print(f"Specificity: {specificity:.4f}")
    print(f"-> Explanation: Out of all actual negative cases, how many did we correctly identify? TN/(TN+FP)\n")
    
    print(f"F1 Score: {f1_score:.4f}")
    print(f"-> Explanation: The harmonic mean of precision and recall, balancing both metrics.\n")


print_classification_metrics(y_test, y_pred_new)

Accuracy: 0.7821
-> Explanation: Overall, how often is the classifier correct? (TP+TN)/Total

Precision: 0.7333
-> Explanation: Out of all positive predictions, how many were actually positive? TP/(TP+FP)

Recall / Sensitivity: 0.7432
-> Explanation: Out of all actual positive cases, how many did we catch? TP/(TP+FN)

Specificity: 0.8095
-> Explanation: Out of all actual negative cases, how many did we correctly identify? TN/(TN+FP)

F1 Score: 0.7383
-> Explanation: The harmonic mean of precision and recall, balancing both metrics.

